# TimesFM — Store Sales Forecasting (Foundation model, bonus)

TimesFM (Google) არის **წინასწარ გაწვრთნილი (pretrained)** დროითი მწკრივების
foundation მოდელი — decoder-only Transformer (~200M პარამეტრი), გაწვრთნილი უზარმაზარ
დროით მონაცემებზე. ჩვენს Walmart მონაცემებზე **საერთოდ არ ვტრენინგებთ**: ვაძლევთ
სერიის ისტორიას context-ად და ვთხოვთ მომდევნო 12 კვირას (**zero-shot**).

**გაშვება ცალკე venv-ში ხდება** (timesfm თავის torch/numpy-ს მოითხოვს):
```bash
python3 -m venv tfm_venv
tfm_venv/bin/pip install "timesfm[torch]" pandas wandb joblib scikit-learn
tfm_venv/bin/python run_timesfm_experiment.py
```

### 1. ბიბლიოთეკები + მონაცემები

In [ ]:
import sys
import os
import warnings

sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
os.environ.setdefault("WANDB_SILENT", "true")

import numpy as np
import timesfm

from src.data import load_raw
from src.metrics import wmae
from src.validation import time_holdout_split
from src.wandb_utils import init_run

train = load_raw("data").train
tr, val = time_holdout_split(train, n_val_weeks=12)
val = val.reset_index(drop=True)

### 2. თითო სერიის ისტორია (context)

მოდელს ვაძლევ თითო სერიის წარსულ `Weekly_Sales`-ს. horizon = holdout-ის კვირების
რაოდენობა (12). ვინახავ, რომელი holdout თარიღი მერამდენე ნაბიჯია, რომ პროგნოზი სწორად
დავამთხვიო.

In [ ]:
holdout_dates = list(np.sort(val["Date"].unique()))
position_of_date = {}
for i, d in enumerate(holdout_dates):
    position_of_date[d] = i
horizon = len(holdout_dates)

# context თითო სერიაზე
history = {}
for key, g in tr.groupby(["Store", "Dept"]):
    y = g.sort_values("Date")["Weekly_Sales"].to_numpy(dtype=np.float32)
    history[key] = y

# მხოლოდ საკმარისი ისტორიის მქონე სერიები
keys = []
inputs = []
for key, y in history.items():
    if len(y) >= 8:
        keys.append(key)
        inputs.append(y)

key_index = {}
for i, key in enumerate(keys):
    key_index[key] = i

global_mean = float(tr["Weekly_Sales"].mean())
print("სერია:", len(keys), "| horizon:", horizon)

### 3. მოდელის ჩატვირთვა (HuggingFace-დან)

In [ ]:
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

### 4. context-ის სიგრძის შემოწმება

zero-shot მოდელს ტრენინგი არ სჭირდება, ამიტომ „ჰიპერპარამეტრი“ = რამდენ წარსულ
წერტილს ვაყურებინებ (`max_context`). ვცდი 4 მნიშვნელობას.

In [ ]:
for ctx in [104, 256, 512, 1024]:
    run = init_run(group="TimesFM_Training", job_type="experiment", name="TimesFM_ctx" + str(ctx),
                   config={"model": "timesfm-2.5-200m", "max_context": ctx,
                           "horizon": horizon, "zero_shot": True})

    model.compile(timesfm.ForecastConfig(max_context=ctx, max_horizon=horizon,
                                         normalize_inputs=True,
                                         use_continuous_quantile_head=True,
                                         infer_is_positive=True))
    point_forecast, _ = model.forecast(horizon=horizon, inputs=inputs)

    # პროგნოზების დამთხვევა val row-ებთან
    preds = np.empty(len(val))
    for i, row in enumerate(val.itertuples()):
        key = (int(row.Store), int(row.Dept))
        j = key_index.get(key)
        if j is None:
            preds[i] = global_mean
        else:
            step = position_of_date[row.Date]
            preds[i] = point_forecast[j][step]
    preds = np.clip(preds, 0, None)

    score = wmae(val["Weekly_Sales"], preds, val["IsHoliday"])
    run.summary["holdout_wmae"] = score
    run.summary["wmae_val"] = score
    run.finish()

    print("context", ctx, "->", round(score, 2), "WMAE")

### შედეგები

| context | WMAE |
|---|---|
| 104 | 1327 |
| **256 / 512 / 1024** | **1309** |

TimesFM zero-shot-მა **1309** მიიღო — **მე-2 ადგილი საერთო ჩარტში, ტრენინგის გარეშე!**
(მხოლოდ LightGBM-ია წინ, 1254). 256/512/1024 ერთნაირია, რადგან სერიებს მაქს. ~143
კვირა აქვთ, ამიტომ დიდი context ისედაც არ ივსება.

**200M მოდელს registry-ში არ ვდებთ** — ის რეპროდუცირებადია `from_pretrained`-ით.